# 01 — Classify a complaint

**Rebuilt 2026-07-24 (ADR-0009):** all classification logic now lives in
`pipeline/classify.py`, the shared module also used by the demo
application. This notebook no longer defines guardrails, redaction,
retrieval, prompt construction, or review-routing logic itself, it
imports `Pipeline` and calls its methods. Fixing something here means
fixing it in the module, not in two places that can drift apart.

Population and batch classification stay in this notebook (they are not
part of the application's job), but the actual classification logic
underneath is identical to what the application calls for its live
demo view.

**Run this after:**
- The workbench is running with the taxonomy AND pipeline-code
  ConfigMaps mounted (manifests/workbench/workbench.yaml,
  ansible/roles/seed_pipeline_code)
- `ansible-playbook ansible/site.yml` has completed
- Optionally, `python3 /opt/app-root/pipeline/smoke_test.py` has passed,
  confirming the pipeline is correctly wired before doing anything else

## Environment variables required

Same as before: `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY`. Everything else
(`LLAMA_STACK_URL`, `GUARDRAILS_URL`, `TAXONOMY_PATH`) has defaults
matching the in-cluster service names; see `pipeline/classify.py`'s
`Config` class if any need overriding.

## Cell 1 — Install dependencies

In [ ]:
import sys
import json
!{sys.executable} -m pip install --quiet requests==2.32.3 pyyaml==6.0.2 minio==7.2.7
print("Dependencies installed.")
# Note: pip may warn about a urllib3 version conflict with appengine-python-standard,
# a package baked into the base workbench image and unrelated to this notebook.
# That warning is expected and harmless; only a raised exception here is a real failure.

## Cell 2 — Import the shared pipeline and run setup

`sys.path` needs the mounted ConfigMap path added before `classify` is
importable, mirrors `smoke_test.py`'s own import. `pipeline.setup()`
replaces what used to be four separate cells (taxonomy load, model
discovery, embedding discovery, vector store creation).

In [ ]:
sys.path.insert(0, "/opt/app-root/pipeline")
from classify import Pipeline

pipeline = Pipeline().setup()

print(f"Taxonomy version {pipeline.taxonomy_version}: "
      f"{len(pipeline.taxonomy['themes'])} themes, "
      f"{len(pipeline.taxonomy['root_causes'])} root causes")
print(f"Confidence threshold: {pipeline.confidence_threshold}")
print(f"Model: {pipeline.model_id}")
print(f"Embedding model: {pipeline.embedding_model_id} ({pipeline.embedding_dimension} dims)")
print(f"Vector store: {pipeline.vector_store_id}")

## Cell 3 — Load complaints, pick one for the single-record walkthrough

In [ ]:
all_complaints = pipeline.load_all_complaints()
print(f"Loaded {len(all_complaints)} complaint records from MinIO.")

complaint = all_complaints[0]
print(f"\nTest complaint: {complaint['complaint_id']}")
print(complaint["body"][:300])

## Cell 4 — Guardrails check and redaction on the test complaint

Just calling into the shared module now, not defining the logic here.

In [ ]:
pii_detected, pii_spans = pipeline.check_pii(complaint["body"])
redacted_body = pipeline.redact_pii(complaint["body"], pii_spans)

print(f"PII detected: {pii_detected} ({len(pii_spans)} matches)")
if pii_detected:
    print(f"Redacted: {redacted_body[:300]}")

## Cell 5 — Single-document round-trip checkpoint

Confirms upload -> attach -> filtered search still works end to end
against this environment before bulk population. Worth keeping as a
fast sanity check on every rebuild, not just a one-time gate.

In [ ]:
test_theme = pipeline.taxonomy["themes"][0]
already_present = test_theme["id"] in pipeline.existing_ids_for_kind("taxonomy")

if already_present:
    print(f"{test_theme['id']} already attached, skipping upload.")
else:
    test_text = f"{test_theme['name']}: {test_theme['definition'].strip()}"
    file_id = pipeline.add_document(test_text, {"kind": "taxonomy", "item_type": "theme", "id": test_theme["id"]})
    print(f"Uploaded and attached: {file_id}")
    import time
    time.sleep(2)

results = pipeline.search_by_kind(test_theme["definition"][:100], "taxonomy")
assert any(r["attributes"]["id"] == test_theme["id"] for r in results), \
    "Round-trip failed: expected document not found in filtered search results"
print("Round-trip confirmed: upload -> attach -> filtered search works.")

## Cell 6 — Populate the taxonomy (17 documents, idempotent)

One document per theme (10) and per root cause (7).

In [ ]:
already = pipeline.existing_ids_for_kind("taxonomy")
added = 0

for t in pipeline.taxonomy["themes"]:
    if t["id"] in already:
        continue
    parts = [f"{t['name']}: {t['definition'].strip()}"]
    if t.get("includes"):
        parts.append("Includes: " + "; ".join(t["includes"]))
    if t.get("excludes"):
        parts.append("Excludes: " + "; ".join(t["excludes"]))
    if t.get("examples"):
        parts.append("Examples: " + "; ".join(t["examples"]))
    pipeline.add_document("\n".join(parts), {"kind": "taxonomy", "item_type": "theme", "id": t["id"]})
    added += 1

for r in pipeline.taxonomy["root_causes"]:
    if r["id"] in already:
        continue
    text = f"{r['name']}: {r['definition'].strip()}"
    pipeline.add_document(text, {"kind": "taxonomy", "item_type": "root_cause", "id": r["id"]})
    added += 1

print(f"Added {added} new taxonomy documents ({len(already)} already present).")

import time
if added:
    time.sleep(3)

final_count = len(pipeline.existing_ids_for_kind("taxonomy"))
expected = len(pipeline.taxonomy["themes"]) + len(pipeline.taxonomy["root_causes"])
print(f"Taxonomy documents in store: {final_count} (expected {expected})")
assert final_count == expected, "Taxonomy population incomplete, check for upload errors above."

## Cell 7 — Populate complaints (200 documents, idempotent, redacted)

Runs a guardrails check and redaction pass on each complaint before
embedding, so the vector store never holds raw PII.

In [ ]:
already = pipeline.existing_ids_for_kind("complaint")
added = 0
redacted_count = 0

for c in all_complaints:
    if c["complaint_id"] in already:
        continue
    detected, spans = pipeline.check_pii(c["body"])
    text_to_embed = pipeline.redact_pii(c["body"], spans) if detected else c["body"]
    if detected:
        redacted_count += 1
    pipeline.add_document(text_to_embed, {
        "kind": "complaint",
        "id": c["complaint_id"],
        "channel": c.get("channel", ""),
        "received_date": c.get("received_date", "")
    })
    added += 1
    if added % 25 == 0:
        print(f"  {added} uploaded...")

print(f"Added {added} new complaint documents ({len(already)} already present), "
      f"{redacted_count} contained PII and were redacted before embedding.")

import time
if added:
    time.sleep(5)

final_count = len(pipeline.existing_ids_for_kind("complaint"))
print(f"Complaint documents in store: {final_count} (expected {len(all_complaints)})")
assert final_count == len(all_complaints), "Complaint population incomplete, check for upload errors above."

## Cell 8 — Classify the test complaint, end to end

One call. Everything that used to be five separate cells (retrieve
context, build prompt, call the model, match the citation, apply
ADR-0004 review routing) now happens inside `classify_complaint()`,
identical to what the application calls for its live demo view.

In [ ]:
record = pipeline.classify_complaint(complaint)
pipeline.write_evidence_record(record)
print(json.dumps(record, indent=2))

## Cell 9 — Batch run over all 200 complaints

Skips already-classified complaints (resumable). Isolates per-record
failures so one bad record does not abort the run. No local function
definitions, everything comes from `pipeline`.

In [ ]:
all_complaints = pipeline.load_all_complaints()  # reload fresh, don't trust a possibly-stale Cell 3 copy

results = {"classified": [], "skipped": [], "failed": []}

for i, c in enumerate(all_complaints):
    if pipeline.already_classified(c["complaint_id"]):
        results["skipped"].append(c["complaint_id"])
        continue
    try:
        record = pipeline.classify_complaint(c)
        pipeline.write_evidence_record(record)
        results["classified"].append(c["complaint_id"])
    except Exception as e:
        results["failed"].append({"complaint_id": c["complaint_id"], "error": str(e)})

    if (i + 1) % 25 == 0:
        print(f"  processed {i + 1}/{len(all_complaints)} "
              f"({len(results['classified'])} classified, "
              f"{len(results['skipped'])} skipped, "
              f"{len(results['failed'])} failed so far)")

print(f"\nDone. Classified: {len(results['classified'])}, "
      f"Skipped (already done): {len(results['skipped'])}, "
      f"Failed: {len(results['failed'])}")

evidence = pipeline.load_all_evidence()
routed_count = sum(1 for r in evidence.values() if r.get("routed_to_review"))
pii_count = sum(1 for r in evidence.values() if r.get("pii_detected"))
print(f"Routed to review: {routed_count}/{len(evidence)}")
print(f"PII detected and redacted: {pii_count}/{len(evidence)}")

if results["failed"]:
    print("\nFailures:")
    for f in results["failed"]:
        print(f"  {f['complaint_id']}: {f['error']}")